The goal of this notebook:
- take a folder that has:
    - initial (repeated) bits with known repeat number
    - final ch1 csv
    - final ch2 csv
- recreate original bit string from initial csv
- sift bit string
- run "ERROR CORRECTION" protocol on sifted bit string
- majority vote to make "best guess" of original

RETURNS:
- best guess final bit string
- post-sift pre-correction error rate
- post correction error rate (between sifted)
- reconstructed error rate (between "majority vote" reconstructed and original)
- number of PUBLIC CHANNEL CALLS

In [1]:
import numpy as np

### PART 1: Data Preparation and Pre-Processing
- load data from file names to create bit string array and channel counts
- convert channel counts to Bob's received bits
- sift Alice's & Bob's arrays to include only bits with conclusive information
    - ALSO create an array that keeps track of indices
        - i.e. which original bit each repeat bit corresponds to.

In [2]:
def load_data(init_file, ch1_file, ch2_file):
    init = np.loadtxt(init_file, delimiter=',', dtype=int)
    ch1 = np.loadtxt(ch1_file, delimiter=',')
    ch2 = np.loadtxt(ch2_file, delimiter=',')
    return init, ch1, ch2

def counts_to_bits(ch1, ch2):
    """
    Returns:
        B_received: array with {0,1,-1}
    """
    B = np.full(len(ch1), -1, dtype=int)

    for i in range(len(ch1)):
        if ch1[i] > ch2[i]:
            B[i] = 1
        elif ch2[i] > ch1[i]:
            B[i] = 0
        else:
            B[i] = -1  # tie → discard

    return B

def create_sifted_index_arrays(A_repeat, B_received, N_duplicate):
    A_sifted = []
    B_sifted = []
    index_array = []

    for i in range(len(B_received)):
        if B_received[i] != -1:
            A_sifted.append(A_repeat[i])
            B_sifted.append(B_received[i])
            index_array.append(i // N_duplicate)

    return np.array(A_sifted), np.array(B_sifted), np.array(index_array)

### PART 2: Error correcting method

I will put all the helper functions for the cascade protocol in a different python file.

You should likewise do the same with Winnow or otherwise :)

In [8]:
from CASCADE import *

### PART 3: Reconstruction
- majority vote protocol to determine original bit string from Bob
- also remake Alice's original string from unaltered duplicate string

In [5]:
def reconstruct_original(B_sifted, index_array, original_length):
    buckets = [[] for _ in range(original_length)]

    for bit, idx in zip(B_sifted, index_array):
        buckets[idx].append(bit)

    result = np.zeros(original_length, dtype=int)

    for i in range(original_length):
        if len(buckets[i]) == 0:
            result[i] = 0
        else:
            result[i] = int(np.round(np.mean(buckets[i])))

    return result

def repeat_to_original(A, N_repeats):
    original_A = np.zeros(len(A)//N_repeats)
    for i in range(0,len(A),N_repeats):
        original_A[i//N_repeats] = A[i]

    return original_A

In [6]:
A = [1,1,1,1,2,2,2,2,3,3,3,3,4,4,4,4]
print(repeat_to_original(A, 4))

[1. 2. 3. 4.]


### PART 3: Error quantification

We want to sample error rates at a few moments in time
- keep_percentage: after sending everything through lasers, how many bits do we keep?
- init_error_rate: after sifting thru "good info", how many errors are there?
- corrected_error_rate: how many errors are there between the duplicated original/corrected strings?
- reconstructed_error_rate: after doing majority vote, how many errors between the un-duplicated and Bob's final string?

In [ ]:
def calculate_error_rate(A, B):
    N = len(A)
    errors = 0
    for i in range(N):
        if A[i]!= B[i]:
            errors+=1
    
    error_rate = errors/N

    return error_rate



In [10]:
def folder_to_data(init_file, ch1_file, ch2_file, N_repeats, error_correction_func):
    """
    INPUTS:
        - init_file : repeated Alice bit string
        - ch1_file : channel 1 of bob
        - ch2_file : channel 2 of bob
        
        - N_repeats: how many times each bit is repeated in Alice's string
        - error_correction_func (callable): bit-flip error correction protocol
            e.g. CASCADE, WINNOW, or NONE
    
    RETURNS:
        - final_string
        - keep_rate
        - init_error_rate
        - corrected_error_rate
        - reconstructed_error_rate
        - public_calls
    """
    alice, ch1, ch2 = load_data(init_file,ch1_file,ch2_file)

    bob = counts_to_bits(ch1,ch2)

    alice_sifted, bob_sifted, index_array= create_sifted_index_arrays(alice, bob, N_repeats)    
    
    keep_rate = len(alice_sifted)/len(alice) # fraction of information we choose to keep
    init_error_rate = calculate_error_rate(alice_sifted, bob_sifted) # pre-correction errors

    bob_corrected, _ = error_correction_func(alice_sifted, bob_sifted)
    corrected_error_rate = calculate_error_rate(alice_sifted,bob_corrected) # post-correction errors

    alice_original = repeat_to_original(alice, N_repeats)
    bob_reconstructed = reconstruct_original(bob_corrected,index_array, len(alice_original))
   

    reconstructed_error_rate = calculate_error_rate(alice_original, bob_reconstructed)

    return bob_reconstructed, keep_rate, init_error_rate, corrected_error_rate, reconstructed_error_rate



In [11]:
bob_reconstructed, keep_rate, init_error_rate, corrected_error_rate, reconstructed_error_rate = folder_to_data("initial_cyrus_bits.csv", "ch1_cyrus.csv", "ch2_cyrus.csv", 16, cascade_correct)

In [12]:
print(f"Keep rate {keep_rate}")
print(f"init {init_error_rate}")
print(f"corrected {corrected_error_rate}")
print(f"reconstructed {reconstructed_error_rate}")

Keep rate 0.24382049560546876
init 0.012631876047467242
corrected 0.002061453080351611
reconstructed 0.002591552734375
